# Phase 3a: OpenVLA-OFT Inference Eval Loop

Closes the loop from a language prompt to a rendered SOARM episode video with
printed success/failure, running **OpenVLA-OFT** as the primary VLA backend
against all 3 frozen `libero_spatial` SOARM tasks (D-08) for a full
3-tasks x 5-10-episodes evaluation (D-09).

## Prerequisites (already confirmed, this notebook does not re-derive them)

- **Phase 1** (`01-colab-env-setup.ipynb`): OpenVLA-OFT loads on Colab GPU in bf16,
  `predict_action` returns an `(8, 7)` action chunk after the `dataset_statistics.json`
  norm_stats overlay (`libero_spatial_no_noops` unnorm_key).
- **Phase 2** (`02-soarm-integration-check.ipynb`): SOARM registered as `Soarm101`,
  3 `libero_spatial` BDDL tasks frozen and crash-free, tuned `eye_in_hand` camera.
- **Phase 3 Plan 01**: shared `LIBERO/libero/libero/vla/` package —
  `VLABackend` protocol, `OFTBackend`, and `eval_loop.run_suite` — this notebook
  is a thin driver over that package, no reimplementation.

## Requirements

- [ ] VLA-01: OpenVLA-OFT loads on Colab GPU (A100 bf16) and produces a valid `(8, 7)`
      action chunk from `(image, language)`.
- [ ] VLA-02: Running the demo cell produces a saved per-episode video file of SOARM
      attempting the task in simulation.
- [ ] VLA-03: Task success/failure printed per episode (BDDL `check_success`/`done`
      protocol) plus an aggregated success-rate summary table across all 3 tasks x
      `EPISODES_PER_TASK` episodes.

## Usage

**BLOCK A** (this block): Run all BLOCK A cells top to bottom, then restart the runtime.
Block A here does **NOT** install anything new — it only re-verifies Phase 1's already-baked
environment (03-RESEARCH.md: "No new installs needed beyond Phase 1's baked environment").

**BLOCK B**: After restart, run the bootstrap + VLA-01/02/03 cells in order.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.


In [ ]:
# Clone/pull the private SoARM-Research repo from GitHub. Uses getpass()
# instead of Colab Secrets — the VS Code Colab extension has no Secrets
# key-icon UI, and getpass() never writes the token into this notebook's
# saved source (unlike a literal hardcoded value, which would leak into git
# history permanently the moment this notebook is committed).
#
# One-time setup: create a GitHub Personal Access Token (Settings > Developer
# settings > Personal access tokens > Fine-grained, scoped to this repo,
# Contents: Read-only). You'll be prompted for it below ONLY on a fresh
# clone (first run of a new Colab VM) — after that, the token is cached in
# this VM's local .git/config for subsequent `git pull`s, and is never
# written anywhere that syncs back to git.
import os
import subprocess
import getpass

REPO_ROOT = "/content/SoARM-Research"

if not os.path.exists(REPO_ROOT):
    GITHUB_TOKEN = getpass.getpass("GitHub token (fine-grained PAT, read-only, scoped to vansh-fyi/SO-ARM-research): ")
    _authed_url = f"https://{GITHUB_TOKEN}@github.com/vansh-fyi/SO-ARM-research.git"
    print(f"Cloning SoARM-Research -> {REPO_ROOT} ...")
    _r = subprocess.run(["git", "clone", "--depth", "1", _authed_url, REPO_ROOT], capture_output=True, text=True)
    del GITHUB_TOKEN, _authed_url  # never keep the token in a live notebook variable longer than needed
else:
    print(f"{REPO_ROOT} already exists — pulling latest ...")
    _r = subprocess.run(["git", "pull"], cwd=REPO_ROOT, capture_output=True, text=True)

if _r.returncode != 0:
    raise RuntimeError(
        "git clone/pull failed — check the token is valid and has read access to "
        "vansh-fyi/SO-ARM-research (the raw git error is withheld here so a bad "
        "token never leaks into cell output)"
    )

_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True).stdout.strip()
print(f"SoARM-Research resolved commit: {_sha}")

In [1]:
import os

# ── Path constants ───────────────────────────────────────────────────────────
REPO_ROOT   = "/content/SoARM-Research"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"                    # path to setup.py directory
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
BDDL_DIR    = f"{LIBERO_ROOT}/bddl_files/libero_spatial"
VIDEO_DIR   = f"{REPO_ROOT}/LIBERO/notebooks/outputs/videos_oft"

os.makedirs(VIDEO_DIR, exist_ok=True)

# ── Frozen libero_spatial SOARM tasks (Phase 2, D-03 resolved in plan 02-04) ──
# Verbatim from explorations/soarm_sanity.py TASKS (lines 65-69) — do not
# re-derive; these 3 filenames are the only ones confirmed within SO101's
# 0.479 m reach from the tuned base.
TASKS = [
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl",
    "pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate.bddl",
    "pick_up_the_black_bowl_on_the_ramekin_and_place_it_on_the_plate.bddl",
]

# Natural-language instruction per task, derived from the BDDL filename itself
# (strip ".bddl", replace underscores with spaces) — no free-text user input
# in this phase (see threat_model Trust Boundaries).
LANGUAGE_MAP = {t: t.replace(".bddl", "").replace("_", " ") for t in TASKS}

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"BDDL_DIR    = {BDDL_DIR}")
print(f"VIDEO_DIR   = {VIDEO_DIR}")
print(f"Saved \u2192 {VIDEO_DIR}  (videos output directory ready)")
print()
print("TASKS:")
for t in TASKS:
    print(f"  - {t}  ->  \"{LANGUAGE_MAP[t]}\"")


REPO_ROOT   = /content/SoARM-Research
LIBERO_PKG  = /content/SoARM-Research/LIBERO
LIBERO_ROOT = /content/SoARM-Research/LIBERO/libero/libero
BDDL_DIR    = /content/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial
VIDEO_DIR   = /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft
Saved → /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft  (videos output directory ready)

TASKS:
  - pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl  ->  "pick up the black bowl from table center and place it on the plate"
  - pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate.bddl  ->  "pick up the black bowl between the plate and the ramekin and place it on the plate"
  - pick_up_the_black_bowl_on_the_ramekin_and_place_it_on_the_plate.bddl  ->  "pick up the black bowl on the ramekin and place it on the plate"


In [2]:
# GPU assertion — OpenVLA-OFT in bf16 needs ~16 GB VRAM; A100 (40 GB) is the target.
# D-05: no 4-bit quantization (bitsandbytes) — bf16 + low_cpu_mem_usage=True on A100.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: The VLA-01 cell will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only / Block A verification on T4.")
else:
    print("A100 confirmed. Proceeding.")


GPU:  NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
A100 confirmed. Proceeding.


## BLOCK A: Install

This notebook requires **NO new pip installs** beyond Phase 1's baked environment
(03-RESEARCH.md Standard Stack Installation section: "No new installs needed beyond
Phase 1's baked environment"). Block A here is a single **verification** cell that
asserts `torch`, `transformers`, and `libero` are all importable and match Phase 1's
pinned versions — it does not re-run any install chain.


In [4]:
# Block A verification — asserts Phase 1's baked environment is present.
# This is a re-verification, NOT a fresh install (03-RESEARCH.md confirms this
# phase only adds eval-loop code on top of Phase 1's environment).
import importlib.metadata
import sys

sys.path.insert(0, LIBERO_PKG)

EXPECTED = {
    "torch":        "2.2.0",
    "transformers": None,   # Phase 1 overrides to a custom fork build; presence-only check
}

all_ok = True
for pkg, expected_ver in EXPECTED.items():
    try:
        installed = importlib.metadata.version(pkg)
        if expected_ver is not None:
            installed_base = installed.split('+')[0]
            ok = installed_base == expected_ver
        else:
            ok = True
        status = "OK" if ok else "MISMATCH"
        if not ok:
            all_ok = False
    except importlib.metadata.PackageNotFoundError:
        installed = "NOT FOUND"
        status = "MISSING"
        all_ok = False
    print(f"{pkg:<15} installed={installed:<20} expected={expected_ver} status={status}")

try:
    import libero  # noqa: F401
    print("libero: importable OK")
except Exception as e:
    all_ok = False
    print(f"libero: IMPORT FAILED — {type(e).__name__}: {e}")

if all_ok:
    print()
    print("Block A verification: PASS \u2014 Phase 1 environment present, no reinstall needed")
else:
    print()
    print("Block A verification: FAIL \u2014 re-run Phase 1's 01-colab-env-setup.ipynb Block A first")


torch           installed=2.2.0+cu121          expected=2.2.0 status=OK
transformers    installed=4.40.1               expected=None status=OK
libero: importable OK

Block A verification: PASS — Phase 1 environment present, no reinstall needed


---

## *** STOP — Restart runtime now ***

Use **Runtime > Restart session** — **NOT** "Disconnect and delete runtime" (the Colab VM
disk persists installs; a full disconnect would lose Phase 1's baked environment and
require a fresh Block A install run from `01-colab-env-setup.ipynb`).

After restarting, re-run Cell 2 (path constants) first, then continue with the Block B
cells below in order.

---


## BLOCK B: Verification (run after restart)

**Critical ordering** — each cell must run in order, matching this project's established
cross-notebook invariant (01-DEBUG-HISTORY.md, referenced in 03-CONTEXT.md canonical_refs):

1. EGL bootstrap (FIRST — no MuJoCo imports before this)
2. LIBERO `~/.libero/config.yaml` bootstrap
3. `sys.path` setup
4. matplotlib `Agg` backend + numba shim
5. VLA-01 — OpenVLA-OFT model load + action shape check
6. VLA-02 / VLA-03 — full eval loop (3 tasks x `EPISODES_PER_TASK` episodes)


In [3]:
# EGL bootstrap — FIRST POST-RESTART CELL
# Creates NVIDIA ICD JSON BEFORE setting MUJOCO_GL env var.
# MUJOCO_GL must be set BEFORE the mujoco/robosuite/libero packages are loaded.
# Run this cell first — before any physics or simulation package is imported.

import os, json

# Step (a): Create NVIDIA EGL ICD JSON
os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
with open("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "w") as _f:
    json.dump({
        "file_format_version": "1.0.0",
        "ICD": {"library_path": "libEGL_nvidia.so.0"}
    }, _f)

# Step (b): Set MuJoCo GL backend env vars
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Step (b2): transformers backend guards — MUST be set before any transformers import.
# Prevents module-level jax/tensorflow imports that crash under this project's numpy 1.26.4 pin.
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

print("EGL ICD created. MUJOCO_GL=egl set. transformers backend guards set (torch only; TF/Flax disabled).")


EGL ICD created. MUJOCO_GL=egl set. transformers backend guards set (torch only; TF/Flax disabled).


In [4]:
# LIBERO config.yaml bootstrap — run BEFORE any import libero
# Pre-creates ~/.libero/config.yaml to prevent input() -> EOFError
# when libero/__init__.py checks for the config file on first import.
# All paths derive from LIBERO_ROOT defined in Cell 2.

import yaml
from pathlib import Path

config = {
    "benchmark_root": LIBERO_ROOT,
    "bddl_files":     f"{LIBERO_ROOT}/bddl_files",
    "init_states":    f"{LIBERO_ROOT}/init_files",
    "datasets":       f"{LIBERO_ROOT}/../datasets",
    "assets":         f"{LIBERO_ROOT}/assets",
}

config_dir = Path.home() / ".libero"
config_dir.mkdir(parents=True, exist_ok=True)
(config_dir / "config.yaml").write_text(yaml.dump(config))
print(f"Saved \u2192 {config_dir / 'config.yaml'}")


Saved → /root/.libero/config.yaml


In [5]:
# sys.path setup — run AFTER config.yaml bootstrap, BEFORE any libero import
# Replicates explorations/soarm_sanity.py / create_scene.py sys.path.insert pattern
# (CLAUDE.md's documented LIBERO.-prefix import quirk).

import sys

if LIBERO_PKG not in sys.path:
    sys.path.insert(0, LIBERO_PKG)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"sys.path[0] = {sys.path[0]}")
print(f"LIBERO_PKG inserted: {LIBERO_PKG}")
print(f"REPO_ROOT inserted: {REPO_ROOT}")


sys.path[0] = /content/SoARM-Research
LIBERO_PKG inserted: /content/SoARM-Research/LIBERO
REPO_ROOT inserted: /content/SoARM-Research


In [6]:
# matplotlib Agg backend (project convention — must precede pyplot import)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Numba compatibility shim — handles any numba/numpy ABI mismatch transparently.
# robosuite uses numba only for JIT optimization; correctness is unaffected if numba is unavailable.
import sys as _sys, types as _types

def _test_numba():
    import numba as _nb
    _nb.njit(lambda x: x + 1)(1.0)  # actually exercise the C extension

try:
    _test_numba()
except Exception as _err:
    class _NumbaStub(_types.ModuleType):
        """No-op stub so robosuite.utils.numba works without a compatible numba."""
        def __init__(self): super().__init__('numba'); self.__version__ = '0.0-stub'
        def njit(self, *a, cache=False, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def jit(self, *a, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def generated_jit(self, *a, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def __getattr__(self, name): return lambda *a, **kw: None
    _sys.modules['numba'] = _NumbaStub()
    print(f"numba stub installed ({type(_err).__name__}: {_err})")
    print("robosuite JIT disabled \u2014 simulation correctness unaffected")
else:
    print("numba OK \u2014 no stub needed")


numba OK — no stub needed


## VLA-01: OpenVLA-OFT Model Load + Action Shape Check

Instantiates `OFTBackend` (Plan 01's shared `vla` package — no reimplementation of the
load here), builds one live SOARM env for `TASKS[0]`, and asserts one real
`predict()` call returns the confirmed `(8, 7)` action chunk.


In [7]:
# VLA-01: OpenVLA-OFT model load + action shape check
import os

from libero.libero.vla import OFTBackend
from libero.libero.envs import OffScreenRenderEnv

try:
    backend = OFTBackend()

    env = OffScreenRenderEnv(
        bddl_file_name=os.path.join(BDDL_DIR, TASKS[0]),
        robots=["Soarm101"],
        camera_heights=256,
        camera_widths=256,
        has_renderer=False,
        has_offscreen_renderer=True,
    )
    obs = env.reset()

    actions = backend.predict(
        {"eye_in_hand": obs["robot0_eye_in_hand_image"]},
        LANGUAGE_MAP[TASKS[0]],
    )

    assert actions.shape == (8, 7), f"expected (8, 7) chunk, got {actions.shape}"
    print(f"VLA-01: PASS \u2014 action shape {actions.shape}, dtype {actions.dtype}")
except Exception as _e:
    import traceback
    traceback.print_exc()
    print(f"VLA-01: FAIL \u2014 {type(_e).__name__}: {_e}")
    raise
finally:
    try:
        env.close()
    except Exception:
        pass


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using LIBERO constants:
  NUM_ACTIONS_CHUNK = 8
  ACTION_DIM = 7
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = NormalizationType.BOUNDS_Q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


07/19 [18:02:53] INFO     | >> NumExpr defaulting to 12 threads.                                       ]8;id=395821;file:///usr/local/lib/python3.12/dist-packages/numexpr/utils.py\utils.py]8;;\:]8;id=430060;file:///usr/local/lib/python3.12/dist-packages/numexpr/utils.py#164\164]8;;\

Loading processor from moojink/openvla-7b-oft-finetuned-libero-spatial...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
<frozen importlib._bootstrap>:530: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.15; use exec_module() instead


Loading model in bf16...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

norm_stats overlaid from dataset_statistics.json: ['libero_spatial_no_noops']
Using unnorm_key: libero_spatial_no_noops
VLA-01: PASS — action shape (8, 7), dtype float64


## VLA-02 / VLA-03: Full OFT Eval Loop — 3 tasks x N episodes

Runs `eval_loop.run_suite` (Plan 01's shared package) across all 3 frozen `libero_spatial`
SOARM tasks (D-08), `EPISODES_PER_TASK` episodes each (within D-09's 5-10 range). Videos
save to `VIDEO_DIR`, one per episode (D-11). `run_suite` prints PASS/FAIL per episode plus
an aggregated success-rate summary table (D-14) internally — this cell adds only the
VLA-02 file-existence proof on top.

Expect this cell to take roughly 15-30 minutes end-to-end on an A100.


In [8]:
# VLA-02 / VLA-03: full eval loop
import os

from libero.libero.vla import run_suite

EPISODES_PER_TASK = 8  # within D-09's 5-10 episodes/task range

env_factory = lambda bddl: OffScreenRenderEnv(
    bddl_file_name=os.path.join(BDDL_DIR, bddl),
    robots=["Soarm101"],
    camera_heights=256,
    camera_widths=256,
    has_renderer=False,
    has_offscreen_renderer=True,
)

results = run_suite(
    env_factory,
    backend,
    TASKS,
    LANGUAGE_MAP,
    episodes_per_task=EPISODES_PER_TASK,
    video_dir=VIDEO_DIR,
    max_steps=600,
)

# VLA-02 proof: at least one episode video file exists under VIDEO_DIR.
_video_paths = [os.path.join(VIDEO_DIR, f"{os.path.basename(t).replace('.bddl', '')}_ep{e}", "video.mp4")
                for t in TASKS for e in range(EPISODES_PER_TASK)]
_existing = [p for p in _video_paths if os.path.exists(p)]
assert len(_existing) > 0, f"No video files found under {VIDEO_DIR}"
print(f"VLA-02: PASS \u2014 {len(_existing)}/{len(_video_paths)} episode video files exist under {VIDEO_DIR}")


Saved videos to /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep0.
Episode 0 (pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate): FAIL (steps=600) -> /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved videos to /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep1.
Episode 1 (pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate): FAIL (steps=600) -> /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep1
Saved videos to /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep2.
Episode 2 (pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate): FAIL (steps=600) -> /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep2
Saved videos to /content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_ep3.
Episode 3 (pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate): FAIL (steps=60

KeyboardInterrupt: 

In [ ]:
from IPython.display import Video
Video("/content/SoARM-Research/LIBERO/notebooks/outputs/videos_oft/pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_ep6/video.mp4", embed=True)


/usr/local/lib/python3.12/dist-packages/IPython/core/history.py:576: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.db.execute("""UPDATE sessions SET end=?, num_cmds=? WHERE


In [ ]:
# Lightweight sanity print on one sample saved video (arrow-notation convention).
import os

_sample = next((p for p in _video_paths if os.path.exists(p)), None)
if _sample:
    _size_kb = os.path.getsize(_sample) / 1024
    print(f"Saved \u2192 {_sample}  ({_size_kb:.1f} KB)")
else:
    print("No sample video found to report size for.")


## Phase 3a Summary

| Requirement | Check | Status | Notes |
|--------------|-------|--------|-------|
| VLA-01: OpenVLA-OFT loads on GPU (A100 bf16), produces valid `(8, 7)` action chunk | VLA-01 cell | ☐ | Update after running |
| VLA-02: Per-episode video file saved for a SOARM task attempt | VLA-02 assertion in eval-loop cell | ☐ | Update after running |
| VLA-03: Per-episode PASS/FAIL + aggregated success-rate table across 3 tasks x `EPISODES_PER_TASK` episodes | eval-loop cell's printed summary table | ☐ | Update after running — fill in aggregate success rate here |

Update the Status column after running all cells top to bottom (post-restart). All three
must show PASS, and at least one saved video must be spot-checked as visually correct
(not black/corrupted), before this phase's Roadmap success criteria #1 and #2 are
considered verified for the OFT backend.

**Phase 3a deliverable:** `LIBERO/notebooks/03a-oft-inference-eval.ipynb` — run Block A,
restart, then Block B cells in order.
